# Phase 1 — dataset, hand-audit, secondary-label investigation, Route A baseline

Launcher for Phase 1 of Laya-Vision (see `docs/PROJECT_SPEC.md` and `DECISIONS.md`).
Code lives in `scripts/`; this notebook only runs it, in order. Gate 1: dataset card + baseline table.

Steps: **P1.1** build the stratified splits · **P1.2** draw the 200-image audit sample (reviewed by hand)
· **P1.4** secondary-label distillation experiments (handwriting / legibility — explored and deferred)
· **P1.5** Route A baseline (VLM describes → Laya decides) on the fixed 2,000-image comparison set.

Runtime: CPU is enough for P1.1–P1.2; P1.4 and P1.5 need a GPU (L4 recommended).
Paths assume the Drive layout in `DECISIONS.md`; adjust `D` for your own account.

In [ ]:
# Phase 1 — setup: mount Drive, clone the public repo. No GPU needed.
import os
os.chdir("/content")  # stand somewhere real (survives a deleted working directory)

from google.colab import drive
drive.mount("/content/drive")

REPO = "/content/laya-vision"
if not os.path.exists(REPO):
    !git clone -q https://github.com/savita10/laya-vision.git {REPO}
else:
    !git -C {REPO} fetch -q origin && git -C {REPO} reset --hard origin/main
os.chdir(REPO)
!git log --oneline -1

!pip install -q "transformers==4.49.0" "laya==0.3.24" datasets pillow

# Drive layout from DECISIONS.md — adjust for your own account
D = "/content/drive/MyDrive/laya-vision"
P1, P2, P3, STEP5 = f"{D}/phase1", f"{D}/phase2", f"{D}/phase3", f"{D}/step5"
for p in (P1, P2, P3): os.makedirs(p, exist_ok=True)

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (fine for this notebook)")

## P1.1 — Build the dataset splits
Stratified 60,000 / 5,000 / 5,000 from RVL-CDIP's own splits, seed 20261003. Writes `splits/*.jsonl` and
`splits_manifest.json` (seed, dataset revision, git hash, per-class counts, SHA-256 of the source label files).
Do a `--limit 10` dry run first; the full run streams the archive (~20–40 min).

In [ ]:
!python scripts/build_dataset.py --out-dir {P1}/splits --limit 10
!head -2 {P1}/splits/train.jsonl

In [ ]:
!python scripts/build_dataset.py --out-dir {P1}/splits
!cat {P1}/splits/splits_manifest.json | head -40

## P1.2 — Audit sample (human review)
Saves 200 stratified train images named `<id>_<label>.png` plus `audit_sheet.csv`. Review by eye and record the
mislabel / unreadable counts in `DECISIONS.md` (result: 193 correct, 4 mislabeled, 3 unreadable). The test split is
frozen from this point.

In [ ]:
!python scripts/audit_sample.py --records {P1}/splits/train.jsonl --n 200 --seed 7 --out {P1}/audit
!ls {P1}/audit | head -5

## P1.4 — Secondary-label distillation (explored, deferred)
Qwen2-VL-2B labels handwriting / legibility. The dry run below was checked against the images by eye; the
investigation (yes-bias, prompt v3, native vs mid-resolution, the L4 cost benchmark) is recorded in `DECISIONS.md`
2026-10-03 and 2026-10-04. **Outcome:** handwriting deprioritised for v1, legibility dropped. Kept here as the record
of how a broken labeller was caught before it touched 60,000 images.

Needs a GPU. Pin `transformers==4.49.0` — 5.x breaks Qwen2-VL internals.

In [ ]:
!pip -q install "transformers==4.49.0" accelerate qwen-vl-utils
import transformers; print("transformers", transformers.__version__)

In [ ]:
# 16-image dry run (the gate that caught the yes-bias)
!python scripts/distill_secondary_labels.py --records {P1}/splits/val.jsonl --limit 16

In [ ]:
# inspect the 16 labels: answer_mass should be ~1; p(true) should rank handwritten pages highest
import json
rows = [json.loads(l) for l in open(f"{P1}/splits/val_secondary.jsonl")]
print(f"{'id':<18}{'hand':>6}{'p(true)':>9}")
for r in sorted(rows, key=lambda r: r["questions"]["handwriting"]["probabilities"]["true"], reverse=True):
    h = r["questions"]["handwriting"]
    print(f'{r["id"]:<18}{str(h["answer"]):>6}{h["probabilities"]["true"]:>9.2f}')

In [ ]:
# save those exact 16 images with the score in the filename, for the eyeball check
ids = ",".join(r["id"] for r in rows)
!python scripts/audit_sample.py --records {P1}/splits/val.jsonl --secondary {P1}/splits/val_secondary.jsonl --ids {ids} --out {P1}/audit_secondary
!ls {P1}/audit_secondary

In [ ]:
# resolution A/B on the same 16 (prompt v3): native vs mid-res (~1.3M px, T4-safe; 2M px OOMs on a T4)
!python scripts/distill_secondary_labels.py --records {P1}/splits/val.jsonl --ids {ids} --out {P1}/exp_v3_native.jsonl
!python scripts/distill_secondary_labels.py --records {P1}/splits/val.jsonl --ids {ids} \
    --min-pixels 1003520 --max-pixels 1254400 --out {P1}/exp_v3_mid.jsonl

In [ ]:
# throughput benchmark that decided the question (0.177 img/s T4, 0.284 img/s L4; batching OOMs)
!python scripts/bench_distill.py --records {P1}/splits/val.jsonl --n 32 --json-out {P1}/bench_distill.json

## P1.5 — Route A baseline
The no-new-training alternative: Qwen2-VL-2B writes a neutral page description (prompt a2, no class names), the
Gate-0 fine-tuned Laya answers the document-type choice zero-shot. Scored on the fixed 2,000-image val comparison set
(`splits/val_comparison_2000_seed11.json`). Dry run with `--limit 32` first (~4 min), then the full run (~4 h on an L4,
resumable). **Result: 0.501 at 7.6 GPU-s/image** — the bar Laya-Vision must beat.

In [ ]:
!pip -q install "laya==0.3.24"
!python scripts/route_a.py --records {P1}/splits/val.jsonl --laya-dir {STEP5}/laya_finetuned \
    --out-dir {P1}/route_a --limit 32

In [ ]:
!python scripts/route_a.py --records {P1}/splits/val.jsonl --laya-dir {STEP5}/laya_finetuned \
    --out-dir {P1}/route_a
!python -c "import json; r=json.load(open('{P1}/route_a/route_a_results.json')); print({k:r[k] for k in ('accuracy','gpu_seconds_per_image') if k in r})"

## Gate 1
Dataset card (`docs/DATASET_CARD.md`), the audit result, the frozen test split and the Route A table are recorded in
`DECISIONS.md`. Next: `phase2_colab_launcher.ipynb` (precompute).